### This notebook demonstrates how to convert a PyTorch model to TensorFlow Lite.

In [10]:
import torch
import sys
import os
import nobuco
from nobuco import ChannelOrder, ChannelOrderingStrategy
from nobuco.converters.node_converter import converter
import tensorflow as tf

sys.path.append(os.path.abspath(".."))
from models.murenn_layer import MuReNNLayer

#### Load the PyTorch model

In [6]:
model = MuReNNLayer(J=2, Q=16, T=32, in_channels=1, J_phi=6, use_conv1d=True, use_power=True)
model.eval()

MuReNNLayer(
  (dtcwt): UDTCWTDirect(
    (fwd_j1): FWD_J1()
    (fwd_j2plus): ModuleList(
      (0): FWD_J2PLUS()
    )
  )
  (conv1d): ParameterList(
      (0): Object of type: Conv1d
      (1): Object of type: Conv1d
    (0): Conv1d(1, 16, kernel_size=(32,), stride=(1,), padding=same, bias=False)
    (1): Conv1d(1, 16, kernel_size=(32,), stride=(1,), padding=same, bias=False)
  )
  (down): Downsampling(
    (phi): DTCWTDirect()
    (relu): ReLU()
  )
  (root): ParameterList(
      (0): Parameter containing: [torch.float32 of size 1x16x1]
      (1): Parameter containing: [torch.float32 of size 1x16x1]
  )
  (sigmoid): Sigmoid()
)

#### Convert Pytorch model to Keras model

In [7]:
# =========================================================
# PyTorch to TensorFlow Lite Conversion: Operator Converters
# =========================================================

# torch.sqrt(x)
# Element-wise square root
@converter(torch.sqrt, channel_ordering_strategy=ChannelOrderingStrategy.FORCE_PYTORCH_ORDER)
def converter_torch_sqrt(x):
    def func(x):
        return tf.math.sqrt(x)
    return func


# x.sqrt()
# Tensor method for element-wise square root
@converter(torch.Tensor.sqrt, channel_ordering_strategy=ChannelOrderingStrategy.FORCE_PYTORCH_ORDER)
def converter_tensor_sqrt(x):
    def func(x):
        return tf.math.sqrt(x)
    return func


# torch.remainder(x, y)
# Element-wise remainder (modulo) operation
@converter(torch.remainder, channel_ordering_strategy=ChannelOrderingStrategy.FORCE_PYTORCH_ORDER)
def converter_torch_remainder(x, y):
    def func(x, y):
        return tf.math.floormod(x, y)
    return func


# x.remainder(y)
# Tensor method for element-wise remainder (modulo)
@converter(torch.Tensor.remainder, channel_ordering_strategy=ChannelOrderingStrategy.FORCE_PYTORCH_ORDER)
def converter_tensor_remainder(x, y):
    def func(x, y):
        return tf.math.floormod(x, y)
    return func


# x.new_zeros(*size, **kwargs)
# Create a new tensor filled with zeros, using the same device/type as x
@converter(torch.Tensor.new_zeros, channel_ordering_strategy=ChannelOrderingStrategy.FORCE_PYTORCH_ORDER)
def converter_tensor_new_zeros(x, *size, **kwargs):
    def func(x, *size, **kwargs):
        # Support different input formats:
        # x.new_zeros(1, 1, 30720)
        # x.new_zeros((1, 1, 30720))
        # x.new_zeros(torch.Size([1, 1, 30720]))
        if len(size) == 1 and isinstance(size[0], (tuple, list, torch.Size)):
            shape = list(size[0])
        else:
            shape = list(size)

        dtype = kwargs.get("dtype", None)
        if dtype is None:
            tf_dtype = x.dtype
        else:
            # Common dtype mapping (extend if needed)
            torch_to_tf = {
                torch.float32: tf.float32,
                torch.float64: tf.float64,
                torch.float16: tf.float16,
                torch.int32: tf.int32,
                torch.int64: tf.int64,
                torch.bool: tf.bool,
                torch.complex64: tf.complex64,
                torch.complex128: tf.complex128,
            }
            tf_dtype = torch_to_tf.get(dtype, x.dtype)

        return tf.zeros(shape=shape, dtype=tf_dtype)

    return func

In [9]:
# =========================================================
# Convert the PyTorch model to a Keras model using the defined converters
# =========================================================
dummy_input = torch.randn(1, 1, 30720)
keras_model = nobuco.pytorch_to_keras(
    model,
    args=[dummy_input],
    inputs_channel_order=ChannelOrder.PYTORCH,
    outputs_channel_order=ChannelOrder.PYTORCH,
)

[Nobuco] Tracing: 0ops [00:00, ?ops/s]/Users/zhang/anaconda3/envs/torch2tf/lib/python3.10/site-packages/nobuco/trace/trace.py:368: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1009.)
  outputs = orig_method(*args, **kwargs)
[Nobuco] Tracing (DONE): : 411ops [00:00, 1527.52ops/s]
[Nobuco] Converting:   6%|▌         | 18/311 [00:00<00:01, 179.63ops/s]

[Nobuco] Converting (DONE):  99%|█████████▉| 309/311 [00:01<00:00, 253.34ops/s]


Legend:
    Green — conversion successful
    Yellow — conversion imprecise
    Red — conversion failed
    Red — no converter found
    Bold — conversion applied directly
    * — subgraph reused
    Tensor — this output is not dependent on any of subgraph's input tensors
    Tensor — this input is a parameter / constant
    Tensor — this tensor is useless

MuReNNLayer[models.murenn_layer](float32_0<1,1,30720>) -> float32_336<1,32,480>
 │  UDTCWTDirect[murenn.dtcwt.udtcwt1d](float32_0<1,1,30720>) -> (float32_34<1,1,30720>, [complex64_16<1,1,30720>, complex64_32<1,1,30720>])
 │   │  FWD_J1[murenn.dtcwt.udtcwt1d](float32_0<1,1,30720>, float32_1<1,1,5>, float32_2<1,1,7>) -> (float32_11<1,2,30720>, float32_12<1,2,30720>)
 │   │   │  repeat[torch.Tensor](float32_1<1,1,5>, 2, 1, 1) -> float32_3<2,1,5>
 │   │   │  repeat[torch.Tensor](float32_2<1,1,7>, 2, 1, 1) -> float32_4<2,1,7>
 │   │   │  pad[torch.nn.functional](float32_0<1,1,30720>, (2, 2)) -> float32_5<1,1,30724>
 │   │   │  pad[torch.

### Convert the Keras model to TensorFlow Lite

In [14]:
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
tflite_model = converter.convert()
with open('murenn_layer.tflite', 'wb') as f:
  f.write(tflite_model)

INFO:tensorflow:Assets written to: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpt0_69_c1/assets


INFO:tensorflow:Assets written to: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpt0_69_c1/assets
2026-03-27 15:40:43.653076: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-27 15:40:43.653087: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-27 15:40:43.653187: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpt0_69_c1
2026-03-27 15:40:43.658941: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-27 15:40:43.658949: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/cf/4zrbxdbx151dcqr0zgx8cqzc0000gp/T/tmpt0_69_c1
2026-03-27 15:40:43.673220: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-03-27 15:40:43.728247: I tensorflow/cc/saved_model/loader.cc:217] Running initialization